In [ ]:
#DISCLAIMER
# =============================================================================
# Academic integrity and AI assistance declaration
# =============================================================================
# This Python workflow was written and developed by the author for the dissertation
# project. Generative AI assistance, specifically ChatGPT, was used as a support
# tool during the coding process.
#
# The AI tool was used to help with debugging, checking consistency of variable names,
# improving code structure, making functions clearer and more reusable, file naming, 
# annotation and identifying possible errors or omissions in the workflow. The 
# underlying calculation logic, project-specific decisions, interpretation of results,
# and final implementation remain the responsibility of the author.
#
# All code outputs used in the dissertation were checked by the author against
# the original Excel-based workflow and inspected for consistency before being
# included in the final report.
#
# The original analysis was run using a local copy of the Excel workbook.
# Personal absolute file paths have been replaced with relative paths for
# reproducibility and privacy.
# =============================================================================

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import griddata


# ============================================================
# 1. Configuration
# ============================================================

# Replace this with the local path to the supplied Excel workbook.
EXCEL_FILE = Path("path/StressCalc_EB4n_AW_v13.xlsx")

# Replace output_file_path with the output folder path for validation tables, filtered datasets and figures.
OUTPUT_DIR = Path("output_file_path")
SHEET_NAME = "FULL_samed0"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Elastic constants used in the original spreadsheet workflow.
# These values should be edited by future users to accurately reflect the material of their experimental data.
E_GPA = 238.0
NU = 0.28

# Excel import settings
SKIPROWS = 6
USECOLS = "B:AD"

COLUMN_NAMES = [
    "point_id_excel", "z_nom", "x_nom", "y_nom", "blank_f",

    "a_long", "da_long", "a0_long", "da0_long",
    "a_trans", "da_trans", "a0_trans", "da0_trans",
    "a_norm", "da_norm", "a0_norm", "da0_norm",

    "eps_long_excel", "deps_long_excel",
    "eps_trans_excel", "deps_trans_excel",
    "eps_norm_excel", "deps_norm_excel",

    "sig_long_excel", "dsig_long_excel",
    "sig_trans_excel", "dsig_trans_excel",
    "sig_norm_excel", "dsig_norm_excel",
]


# ============================================================
# 2. Data loading
# ============================================================

def load_full_samed0(path: str | Path) -> pd.DataFrame:
    """
    Loads the integrated FULL_samed0 worksheet into a point-based dataframe.

    Missing coordinate, lattice-parameter, reference-value or uncertainty values
    are preserved at import stage so that they can be flagged, reported and
    excluded only at the correct validation/filtering stage.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Workbook not found at: {path}")

    df = pd.read_excel(
        path,
        sheet_name=SHEET_NAME,
        skiprows=SKIPROWS,
        usecols=USECOLS,
        header=None,
        names=COLUMN_NAMES,
        engine="openpyxl",
    )

    # Preserve partially missing rows; only remove rows that are completely empty.
    df = df.dropna(how="all").copy()

    df["excel_row"] = df.index + SKIPROWS + 1
    df["point_id_py"] = range(len(df))

    df = df.drop(columns=["blank_f"])

    for col in df.columns:
        if col != "point_id_excel":
            df[col] = pd.to_numeric(df[col], errors="coerce")

    front_cols = ["point_id_py", "point_id_excel", "excel_row", "z_nom", "x_nom", "y_nom"]
    other_cols = [c for c in df.columns if c not in front_cols]
    df = df[front_cols + other_cols]

    return df


# ============================================================
# 3. Calculation functions with missing-data safety
# ============================================================

def calc_strain_safe(a: float, da: float, a0: float, da0: float) -> tuple:
    """
    Calculates strain and propagated strain uncertainty safely.

    Strain equation:
        eps = ((a - a0) / a0) * 1e6

    Uncertainty equation:
        deps = (eps + 1e6) * sqrt((da/a)^2 + (da0/a0)^2)

    Returns:
        eps_microstrain,
        deps_microstrain,
        strain_calc_valid,
        strain_uncertainty_valid,
        strain_calc_issue
    """
    if pd.isna(a) or pd.isna(a0):
        return np.nan, np.nan, False, False, "missing_measured_or_reference_lattice_parameter"

    if a == 0 or a0 == 0:
        return np.nan, np.nan, False, False, "zero_lattice_parameter_or_reference"

    eps = ((a - a0) / a0) * 1.0e6

    if pd.isna(da) or pd.isna(da0):
        return eps, np.nan, True, False, "missing_lattice_parameter_uncertainty"

    if da < 0 or da0 < 0:
        return eps, np.nan, True, False, "negative_lattice_parameter_uncertainty"

    deps = (eps + 1.0e6) * math.sqrt((da / a) ** 2 + (da0 / a0) ** 2)

    return eps, deps, True, True, "ok"


def calc_stresses_safe(
    eps_long: float,
    eps_trans: float,
    eps_norm: float,
    deps_long: float,
    deps_trans: float,
    deps_norm: float,
    e_gpa: float = E_GPA,
    nu: float = NU,
) -> tuple:
    """
    Reproduce the Excel isotropic stress reconstruction and propagated uncertainty,
    while preserving missing-data cases instead of crashing.

    Stress values are returned in MPa.

    Returns:
        sig_long, dsig_long,
        sig_trans, dsig_trans,
        sig_norm, dsig_norm,
        stress_calc_valid,
        stress_uncertainty_valid,
        stress_calc_issue
    """
    eps_values = [eps_long, eps_trans, eps_norm]

    if any(pd.isna(v) for v in eps_values):
        return (
            np.nan, np.nan,
            np.nan, np.nan,
            np.nan, np.nan,
            False, False,
            "missing_directional_strain",
        )

    hydro = nu / (1.0 - 2.0 * nu)
    prefactor = e_gpa / (1.0 + nu)

    trace_eps = eps_long + eps_trans + eps_norm

    sig_long = prefactor * (eps_long + hydro * trace_eps) / 1000.0
    sig_trans = prefactor * (eps_trans + hydro * trace_eps) / 1000.0
    sig_norm = prefactor * (eps_norm + hydro * trace_eps) / 1000.0

    deps_values = [deps_long, deps_trans, deps_norm]

    if any(pd.isna(v) for v in deps_values):
        return (
            sig_long, np.nan,
            sig_trans, np.nan,
            sig_norm, np.nan,
            True, False,
            "stress_calculated_but_uncertainty_unavailable",
        )

    dsig_long = prefactor * math.sqrt(
        hydro * deps_long**2
        + hydro**2 * (deps_long**2 + deps_trans**2 + deps_norm**2)
    ) / 1000.0

    dsig_trans = prefactor * math.sqrt(
        hydro * deps_trans**2
        + hydro**2 * (deps_long**2 + deps_trans**2 + deps_norm**2)
    ) / 1000.0

    dsig_norm = prefactor * math.sqrt(
        hydro * deps_norm**2
        + hydro**2 * (deps_long**2 + deps_trans**2 + deps_norm**2)
    ) / 1000.0

    return (
        sig_long, dsig_long,
        sig_trans, dsig_trans,
        sig_norm, dsig_norm,
        True, True,
        "ok",
    )


# ============================================================
# 4. Main numerical pipeline
# ============================================================

def run_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate Python strain, strain uncertainty, stress and stress uncertainty outputs.

    Missing or incomplete rows are preserved and flagged. The workflow calculates
    whatever is physically possible and marks unavailable quantities as NaN.
    """
    results = df.copy()

    strain_map = [
        ("long", "a_long", "da_long", "a0_long", "da0_long"),
        ("trans", "a_trans", "da_trans", "a0_trans", "da0_trans"),
        ("norm", "a_norm", "da_norm", "a0_norm", "da0_norm"),
    ]

    for prefix, a_col, da_col, a0_col, da0_col in strain_map:
        vals = results.apply(
            lambda row: calc_strain_safe(
                row[a_col],
                row[da_col],
                row[a0_col],
                row[da0_col],
            ),
            axis=1,
            result_type="expand",
        )

        vals.columns = [
            f"eps_{prefix}_py",
            f"deps_{prefix}_py",
            f"strain_{prefix}_calc_valid",
            f"strain_{prefix}_uncertainty_valid",
            f"strain_{prefix}_calc_issue",
        ]

        results = pd.concat([results, vals], axis=1)

    stress_vals = results.apply(
        lambda row: calc_stresses_safe(
            row["eps_long_py"],
            row["eps_trans_py"],
            row["eps_norm_py"],
            row["deps_long_py"],
            row["deps_trans_py"],
            row["deps_norm_py"],
        ),
        axis=1,
        result_type="expand",
    )

    stress_vals.columns = [
        "sig_long_py", "dsig_long_py",
        "sig_trans_py", "dsig_trans_py",
        "sig_norm_py", "dsig_norm_py",
        "stress_calc_valid",
        "stress_uncertainty_valid",
        "stress_calc_issue",
    ]

    results = pd.concat([results, stress_vals], axis=1)

    return results


# ============================================================
# 5. Validation helpers
# ============================================================

def add_validation_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add point-by-point Excel vs Python comparison columns.
    """
    out = df.copy()

    compare_pairs = [
        ("eps_long", "eps_long_excel", "eps_long_py"),
        ("deps_long", "deps_long_excel", "deps_long_py"),
        ("eps_trans", "eps_trans_excel", "eps_trans_py"),
        ("deps_trans", "deps_trans_excel", "deps_trans_py"),
        ("eps_norm", "eps_norm_excel", "eps_norm_py"),
        ("deps_norm", "deps_norm_excel", "deps_norm_py"),
        ("sig_long", "sig_long_excel", "sig_long_py"),
        ("dsig_long", "dsig_long_excel", "dsig_long_py"),
        ("sig_trans", "sig_trans_excel", "sig_trans_py"),
        ("dsig_trans", "dsig_trans_excel", "dsig_trans_py"),
        ("sig_norm", "sig_norm_excel", "sig_norm_py"),
        ("dsig_norm", "dsig_norm_excel", "dsig_norm_py"),
    ]

    for label, excel_col, py_col in compare_pairs:
        out[f"{label}_diff"] = out[py_col] - out[excel_col]
        out[f"{label}_abs_diff"] = out[f"{label}_diff"].abs()

    return out


def print_structural_summary(df: pd.DataFrame) -> None:
    print("Structural validation:")
    print(f"  Number of points loaded: {len(df)}")
    print(f"  Missing x_nom values: {df['x_nom'].isna().sum()}")
    print(f"  Missing y_nom values: {df['y_nom'].isna().sum()}")
    print(f"  Missing a_long values: {df['a_long'].isna().sum()}")
    print(f"  Missing a_trans values: {df['a_trans'].isna().sum()}")
    print(f"  Missing a_norm values: {df['a_norm'].isna().sum()}")


def inspect_point_structure(df: pd.DataFrame) -> None:
    print("\nPoint structure inspection:")
    print(f"  Total rows: {len(df)}")
    print(f"  Unique (x_nom, y_nom) pairs: {df[['x_nom', 'y_nom']].drop_duplicates().shape[0]}")
    print(f"  Unique (x_nom, y_nom, z_nom) triples: {df[['x_nom', 'y_nom', 'z_nom']].drop_duplicates().shape[0]}")

    dup_xy = df[df.duplicated(subset=["x_nom", "y_nom"], keep=False)]
    if len(dup_xy) > 0:
        print("\n  Repeated (x_nom, y_nom) pairs:")
        print(
            dup_xy[
                ["point_id_py", "point_id_excel", "excel_row", "x_nom", "y_nom", "z_nom"]
            ].sort_values(["x_nom", "y_nom", "z_nom"])
        )
    else:
        print("\n  No repeated (x_nom, y_nom) pairs found.")

    dup_xyz = df[df.duplicated(subset=["x_nom", "y_nom", "z_nom"], keep=False)]
    if len(dup_xyz) > 0:
        print("\n  True duplicate (x_nom, y_nom, z_nom) triples found:")
        print(
            dup_xyz[
                ["point_id_py", "point_id_excel", "excel_row", "x_nom", "y_nom", "z_nom"]
            ].sort_values(["x_nom", "y_nom", "z_nom"])
        )
    else:
        print("\n  No true duplicate (x_nom, y_nom, z_nom) triples found.")


def inspect_duplicate_content(df: pd.DataFrame) -> None:
    """
    Inspect duplicated coordinate rows to determine whether they are exact duplicates.
    """
    dup = df[df.duplicated(subset=["x_nom", "y_nom", "z_nom"], keep=False)].copy()

    if dup.empty:
        print("\nNo duplicated nominal locations found.")
        return

    print("\nDuplicate content inspection:")

    show_cols = [
        "point_id_py", "point_id_excel", "excel_row",
        "x_nom", "y_nom", "z_nom",
        "a_long", "da_long",
        "a_trans", "da_trans",
        "a_norm", "da_norm",
        "eps_long_excel", "eps_trans_excel", "eps_norm_excel",
        "sig_long_excel", "sig_trans_excel", "sig_norm_excel",
    ]

    print(dup[show_cols])

    if len(dup) == 2:
        non_id_cols = [
            c for c in dup.columns
            if c not in ["point_id_py", "point_id_excel", "excel_row"]
        ]

        identical = True

        for col in non_id_cols:
            v1 = dup.iloc[0][col]
            v2 = dup.iloc[1][col]

            if pd.isna(v1) and pd.isna(v2):
                continue

            if v1 != v2:
                identical = False
                print(f"\nColumn differs: {col}")
                print(f"  Row {dup.iloc[0]['excel_row']}: {v1}")
                print(f"  Row {dup.iloc[1]['excel_row']}: {v2}")

        if identical:
            print("\nThese duplicated nominal-location rows are exact duplicates in content.")
        else:
            print("\nThese duplicated nominal-location rows are NOT exact duplicates in content.")


def print_validation_summary(df: pd.DataFrame) -> None:
    """
    Print Excel-to-Python validation summary while tolerating missing-data rows.
    """
    keys = [
        "eps_long_abs_diff", "deps_long_abs_diff",
        "eps_trans_abs_diff", "deps_trans_abs_diff",
        "eps_norm_abs_diff", "deps_norm_abs_diff",
        "sig_long_abs_diff", "dsig_long_abs_diff",
        "sig_trans_abs_diff", "dsig_trans_abs_diff",
        "sig_norm_abs_diff", "dsig_norm_abs_diff",
    ]

    print("\nNumerical validation summary:")
    for key in keys:
        valid_values = df[key].dropna()

        if valid_values.empty:
            print(f"  {key}: no valid comparison values available")
        else:
            print(
                f"  {key}: max = {valid_values.max():.12f}, "
                f"mean = {valid_values.mean():.12f}, "
                f"NaN comparisons = {df[key].isna().sum()}"
            )


# ============================================================
# 6. Repeated-location handling
# ============================================================

def flag_repeated_nominal_locations(df: pd.DataFrame) -> pd.DataFrame:
    """
    Flag repeated nominal locations.
    """
    out = df.copy()
    out["repeat_xy"] = out.duplicated(subset=["x_nom", "y_nom"], keep=False)
    out["repeat_xyz"] = out.duplicated(subset=["x_nom", "y_nom", "z_nom"], keep=False)
    return out


def build_analysis_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a cleaned dataset for analysis and plotting.

    Exact duplicated nominal-location rows are removed.
    The full validation dataset remains unchanged.
    """
    out = flag_repeated_nominal_locations(df.copy())

    comparison_cols = [
        "x_nom", "y_nom", "z_nom",
        "a_long", "da_long", "a0_long", "da0_long",
        "a_trans", "da_trans", "a0_trans", "da0_trans",
        "a_norm", "da_norm", "a0_norm", "da0_norm",
        "eps_long_py", "deps_long_py",
        "eps_trans_py", "deps_trans_py",
        "eps_norm_py", "deps_norm_py",
        "sig_long_py", "dsig_long_py",
        "sig_trans_py", "dsig_trans_py",
        "sig_norm_py", "dsig_norm_py",
    ]

    out_clean = out.drop_duplicates(subset=comparison_cols, keep="first").copy()
    out_clean["used_for_analysis"] = True

    return out_clean


# ============================================================
# 7. Quality filtering and missing-data diagnostics
# ============================================================

def add_quality_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add explicit quality-control flags for interpretation and plotting.

    The workflow distinguishes between:
    - missing coordinates,
    - missing lattice/reference values,
    - missing uncertainty inputs,
    - unavailable strain/stress calculations,
    - unavailable propagated uncertainty,
    - catastrophic uncertainty values.
    """
    out = df.copy()

    coordinate_cols = ["x_nom", "y_nom"]

    lattice_value_cols = [
        "a_long", "a0_long",
        "a_trans", "a0_trans",
        "a_norm", "a0_norm",
    ]

    uncertainty_input_cols = [
        "da_long", "da0_long",
        "da_trans", "da0_trans",
        "da_norm", "da0_norm",
    ]

    stress_output_cols = ["sig_long_py", "sig_trans_py", "sig_norm_py"]
    stress_unc_cols = ["dsig_long_py", "dsig_trans_py", "dsig_norm_py"]

    out["missing_coordinate_flag"] = out[coordinate_cols].isna().any(axis=1)
    out["missing_lattice_value_flag"] = out[lattice_value_cols].isna().any(axis=1)
    out["missing_uncertainty_input_flag"] = out[uncertainty_input_cols].isna().any(axis=1)

    out["strain_calc_unavailable_flag"] = ~(
        out["strain_long_calc_valid"]
        & out["strain_trans_calc_valid"]
        & out["strain_norm_calc_valid"]
    )

    out["strain_uncertainty_unavailable_flag"] = ~(
        out["strain_long_uncertainty_valid"]
        & out["strain_trans_uncertainty_valid"]
        & out["strain_norm_uncertainty_valid"]
    )

    out["stress_calc_unavailable_flag"] = ~out["stress_calc_valid"]
    out["stress_uncertainty_unavailable_flag"] = ~out["stress_uncertainty_valid"]

    out["missing_input_flag"] = (
        out["missing_coordinate_flag"]
        | out["missing_lattice_value_flag"]
        | out["missing_uncertainty_input_flag"]
    )

    out["nan_output_flag"] = (
        out[stress_output_cols].isna().any(axis=1)
        | out[stress_unc_cols].isna().any(axis=1)
    )

    out = flag_repeated_nominal_locations(out)

    def high_uncertainty_flag(series: pd.Series, quantile: float = 0.90) -> pd.Series:
        valid = series.dropna()
        if valid.empty:
            return pd.Series(False, index=series.index)
        threshold = valid.quantile(quantile)
        return series > threshold

    out["high_unc_long_flag"] = high_uncertainty_flag(out["dsig_long_py"])
    out["high_unc_trans_flag"] = high_uncertainty_flag(out["dsig_trans_py"])
    out["high_unc_norm_flag"] = high_uncertainty_flag(out["dsig_norm_py"])

    out["high_unc_any_flag"] = (
        out["high_unc_long_flag"]
        | out["high_unc_trans_flag"]
        | out["high_unc_norm_flag"]
    )

    out["catastrophic_da_norm_flag"] = out["da_norm"] > 1.0e-3

    out["catastrophic_unc_long_flag"] = out["dsig_long_py"] > 1000.0
    out["catastrophic_unc_trans_flag"] = out["dsig_trans_py"] > 1000.0
    out["catastrophic_unc_norm_flag"] = out["dsig_norm_py"] > 1000.0

    out["catastrophic_unc_any_flag"] = (
        out["catastrophic_unc_long_flag"]
        | out["catastrophic_unc_trans_flag"]
        | out["catastrophic_unc_norm_flag"]
        | out["catastrophic_da_norm_flag"]
    )

    out["used_for_plotting"] = ~(
        out["missing_coordinate_flag"]
        | out["stress_calc_unavailable_flag"]
        | out[stress_output_cols].isna().any(axis=1)
    )

    out["used_for_interpretation"] = (
        out["used_for_plotting"]
        & ~out["stress_uncertainty_unavailable_flag"]
        & ~out["catastrophic_unc_any_flag"]
    )

    return out


def print_quality_flag_summary(df: pd.DataFrame) -> None:
    print("\nQuality flag summary:")
    print(f"  Missing-coordinate rows:             {df['missing_coordinate_flag'].sum()}")
    print(f"  Missing lattice/reference rows:      {df['missing_lattice_value_flag'].sum()}")
    print(f"  Missing uncertainty-input rows:      {df['missing_uncertainty_input_flag'].sum()}")
    print(f"  Strain unavailable rows:             {df['strain_calc_unavailable_flag'].sum()}")
    print(f"  Stress unavailable rows:             {df['stress_calc_unavailable_flag'].sum()}")
    print(f"  Stress uncertainty unavailable rows: {df['stress_uncertainty_unavailable_flag'].sum()}")
    print(f"  Repeated xyz rows:                   {df['repeat_xyz'].sum()}")
    print(f"  Relative high-unc rows(any):         {df['high_unc_any_flag'].sum()}")
    print(f"  Catastrophic da_norm rows:           {df['catastrophic_da_norm_flag'].sum()}")
    print(f"  Catastrophic unc rows(any):          {df['catastrophic_unc_any_flag'].sum()}")
    print(f"  Rows used for plotting:              {df['used_for_plotting'].sum()}")
    print(f"  Rows used for interpretation:        {df['used_for_interpretation'].sum()}")


def export_missing_data_reports(df: pd.DataFrame, output_dir: Path) -> None:
    """
    Export missing-data diagnostics and a compact missing-data summary.

    These outputs support future datasets where missing rows or missing information
    may occur. Missing data are not silently filled.
    """
    diagnostic_cols = [
        "point_id_py", "point_id_excel", "excel_row",
        "x_nom", "y_nom", "z_nom",
        "missing_coordinate_flag",
        "missing_lattice_value_flag",
        "missing_uncertainty_input_flag",
        "strain_long_calc_valid",
        "strain_trans_calc_valid",
        "strain_norm_calc_valid",
        "strain_long_uncertainty_valid",
        "strain_trans_uncertainty_valid",
        "strain_norm_uncertainty_valid",
        "stress_calc_valid",
        "stress_uncertainty_valid",
        "strain_long_calc_issue",
        "strain_trans_calc_issue",
        "strain_norm_calc_issue",
        "stress_calc_issue",
        "used_for_plotting",
        "used_for_interpretation",
    ]

    available_cols = [c for c in diagnostic_cols if c in df.columns]

    diagnostics = df[available_cols].copy()
    diagnostics.to_csv(output_dir / "missing_data_diagnostics.csv", index=False)

    summary_items = {
        "total_rows": len(df),
        "missing_coordinate_rows": int(df["missing_coordinate_flag"].sum()),
        "missing_lattice_or_reference_rows": int(df["missing_lattice_value_flag"].sum()),
        "missing_uncertainty_input_rows": int(df["missing_uncertainty_input_flag"].sum()),
        "strain_calculation_unavailable_rows": int(df["strain_calc_unavailable_flag"].sum()),
        "stress_calculation_unavailable_rows": int(df["stress_calc_unavailable_flag"].sum()),
        "stress_uncertainty_unavailable_rows": int(df["stress_uncertainty_unavailable_flag"].sum()),
        "rows_used_for_plotting": int(df["used_for_plotting"].sum()),
        "rows_used_for_interpretation": int(df["used_for_interpretation"].sum()),
    }

    summary = pd.DataFrame(
        [{"check": key, "count": value} for key, value in summary_items.items()]
    )

    summary.to_csv(output_dir / "missing_data_summary.csv", index=False)

    print("\nSaved missing-data diagnostics:")
    print(f"  {output_dir / 'missing_data_diagnostics.csv'}")
    print(f"  {output_dir / 'missing_data_summary.csv'}")


def inspect_catastrophic_rows(df: pd.DataFrame) -> None:
    """
    Print rows excluded from interpretation due to catastrophic uncertainty.
    """
    bad = df[df["catastrophic_unc_any_flag"]].copy()

    if bad.empty:
        print("\nNo catastrophic-uncertainty rows found.")
        return

    print("\nCatastrophic-uncertainty rows:")

    cols = [
        "point_id_py", "excel_row", "x_nom", "y_nom",
        "da_long", "da_trans", "da_norm",
        "dsig_long_py", "dsig_trans_py", "dsig_norm_py",
        "catastrophic_da_norm_flag",
        "catastrophic_unc_long_flag",
        "catastrophic_unc_trans_flag",
        "catastrophic_unc_norm_flag",
    ]

    print(bad[cols].sort_values(["y_nom", "x_nom"]))


def export_rejected_points(df: pd.DataFrame, output_dir: Path) -> None:
    """
    Export points excluded from interpretation.
    """
    rejected = df[~df["used_for_interpretation"]].copy()
    rejected.to_csv(output_dir / "rejected_points_for_interpretation.csv", index=False)


def print_interpretable_points_by_y(df: pd.DataFrame) -> None:
    """
    Print how many interpretable points remain on each y-line.
    """
    counts = (
        df[df["used_for_interpretation"]]
        .groupby("y_nom")
        .size()
        .reset_index(name="interpretable_points")
        .sort_values("y_nom")
    )

    print("\nInterpretable points by y_nom:")
    print(counts)


# ============================================================
# 8. Reference-value comparison tables
# ============================================================

def build_reference_value_comparison_table(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    retained_df = df[df["used_for_interpretation"]].copy()

    if retained_df.empty:
        raise ValueError("No retained interpretation points available for reference-value comparison.")

    components = [
        {"direction": "Longitudinal", "a_col": "a_long", "a0_col": "a0_long", "eps_col": "eps_long_py"},
        {"direction": "Transverse", "a_col": "a_trans", "a0_col": "a0_trans", "eps_col": "eps_trans_py"},
        {"direction": "Normal", "a_col": "a_norm", "a0_col": "a0_norm", "eps_col": "eps_norm_py"},
    ]

    rows = []

    for item in components:
        direction = item["direction"]
        a_col = item["a_col"]
        a0_col = item["a0_col"]
        eps_col = item["eps_col"]

        delta_a = retained_df[a_col] - retained_df[a0_col]
        relative_delta_microstrain = (delta_a / retained_df[a0_col]) * 1.0e6

        n_above_a0 = (delta_a > 0).sum()
        n_below_a0 = (delta_a < 0).sum()
        n_equal_a0 = (delta_a == 0).sum()

        max_delta_idx = delta_a.idxmax()
        min_delta_idx = delta_a.idxmin()
        max_abs_delta_idx = delta_a.abs().idxmax()

        max_delta_row = retained_df.loc[max_delta_idx]
        min_delta_row = retained_df.loc[min_delta_idx]
        max_abs_delta_row = retained_df.loc[max_abs_delta_idx]

        rows.append(
            {
                "direction": direction,
                "n_retained_points": len(retained_df),
                "mean_a": retained_df[a_col].mean(),
                "mean_a0": retained_df[a0_col].mean(),
                "mean_delta_a": delta_a.mean(),
                "median_delta_a": delta_a.median(),
                "mean_abs_delta_a": delta_a.abs().mean(),
                "min_delta_a": delta_a.min(),
                "x_at_min_delta_a_mm": min_delta_row["x_nom"],
                "y_at_min_delta_a_mm": min_delta_row["y_nom"],
                "max_delta_a": delta_a.max(),
                "x_at_max_delta_a_mm": max_delta_row["x_nom"],
                "y_at_max_delta_a_mm": max_delta_row["y_nom"],
                "max_abs_delta_a": delta_a.loc[max_abs_delta_idx],
                "max_abs_delta_a_magnitude": abs(delta_a.loc[max_abs_delta_idx]),
                "x_at_max_abs_delta_a_mm": max_abs_delta_row["x_nom"],
                "y_at_max_abs_delta_a_mm": max_abs_delta_row["y_nom"],
                "mean_relative_delta_microstrain": relative_delta_microstrain.mean(),
                "median_relative_delta_microstrain": relative_delta_microstrain.median(),
                "mean_strain_microstrain": retained_df[eps_col].mean(),
                "median_strain_microstrain": retained_df[eps_col].median(),
                "n_a_above_a0": n_above_a0,
                "n_a_below_a0": n_below_a0,
                "n_a_equal_a0": n_equal_a0,
                "fraction_a_above_a0": n_above_a0 / len(retained_df),
                "fraction_a_below_a0": n_below_a0 / len(retained_df),
            }
        )

    ref_table = pd.DataFrame(rows)

    output_path = output_dir / "reference_value_comparison_table.csv"
    ref_table.to_csv(output_path, index=False)

    print("\nReference-value comparison table:")
    print(ref_table.to_string(index=False))
    print(f"\nSaved reference-value comparison table to: {output_path}")

    return ref_table


def build_main_report_reference_table(ref_table: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    report = ref_table.copy()

    report["fraction_a_above_a0_percent"] = report["fraction_a_above_a0"] * 100.0
    report["fraction_a_below_a0_percent"] = report["fraction_a_below_a0"] * 100.0

    report["min_delta_location_mm"] = (
        "("
        + report["x_at_min_delta_a_mm"].round(3).astype(str)
        + ", "
        + report["y_at_min_delta_a_mm"].round(3).astype(str)
        + ")"
    )

    report["max_delta_location_mm"] = (
        "("
        + report["x_at_max_delta_a_mm"].round(3).astype(str)
        + ", "
        + report["y_at_max_delta_a_mm"].round(3).astype(str)
        + ")"
    )

    main_cols = [
        "direction",
        "n_retained_points",
        "mean_delta_a",
        "median_delta_a",
        "min_delta_a",
        "min_delta_location_mm",
        "max_delta_a",
        "max_delta_location_mm",
        "mean_relative_delta_microstrain",
        "fraction_a_above_a0_percent",
        "fraction_a_below_a0_percent",
    ]

    report = report[main_cols].copy()

    output_path = output_dir / "main_report_reference_value_comparison_table.csv"
    report.to_csv(output_path, index=False)

    print("\nMain-report reference-value comparison table:")
    print(report.to_string(index=False))
    print(f"\nSaved main-report reference-value comparison table to: {output_path}")

    return report


# ============================================================
# 9. Directional strain summary tables
# ============================================================

def build_directional_strain_summary_table(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    retained_df = df[df["used_for_interpretation"]].copy()

    if retained_df.empty:
        raise ValueError("No retained interpretation points available for strain summary.")

    n_total_analysis_points = len(df)
    n_retained_points = len(retained_df)
    n_rejected_points = n_total_analysis_points - n_retained_points
    retained_fraction = n_retained_points / n_total_analysis_points

    components = [
        {"direction": "Longitudinal", "strain_col": "eps_long_py", "unc_col": "deps_long_py"},
        {"direction": "Transverse", "strain_col": "eps_trans_py", "unc_col": "deps_trans_py"},
        {"direction": "Normal", "strain_col": "eps_norm_py", "unc_col": "deps_norm_py"},
    ]

    rows = []

    for item in components:
        direction = item["direction"]
        strain_col = item["strain_col"]
        unc_col = item["unc_col"]

        strains = retained_df[strain_col]
        uncertainties = retained_df[unc_col]

        max_idx = strains.idxmax()
        min_idx = strains.idxmin()
        max_abs_idx = strains.abs().idxmax()

        max_row = retained_df.loc[max_idx]
        min_row = retained_df.loc[min_idx]
        max_abs_row = retained_df.loc[max_abs_idx]

        n_tensile = (strains > 0).sum()
        n_compressive = (strains < 0).sum()
        n_zero = (strains == 0).sum()

        q25 = strains.quantile(0.25)
        q75 = strains.quantile(0.75)

        rows.append(
            {
                "direction": direction,
                "n_total_analysis_points": n_total_analysis_points,
                "n_retained_points": n_retained_points,
                "n_rejected_points": n_rejected_points,
                "retained_fraction": retained_fraction,
                "min_strain_microstrain": strains.min(),
                "min_x_nom_mm": min_row["x_nom"],
                "min_y_nom_mm": min_row["y_nom"],
                "uncertainty_at_min_microstrain": min_row[unc_col],
                "max_strain_microstrain": strains.max(),
                "max_x_nom_mm": max_row["x_nom"],
                "max_y_nom_mm": max_row["y_nom"],
                "uncertainty_at_max_microstrain": max_row[unc_col],
                "max_abs_strain_microstrain": max_abs_row[strain_col],
                "max_abs_strain_magnitude_microstrain": abs(max_abs_row[strain_col]),
                "max_abs_x_nom_mm": max_abs_row["x_nom"],
                "max_abs_y_nom_mm": max_abs_row["y_nom"],
                "uncertainty_at_max_abs_microstrain": max_abs_row[unc_col],
                "strain_range_microstrain": strains.max() - strains.min(),
                "mean_strain_microstrain": strains.mean(),
                "median_strain_microstrain": strains.median(),
                "std_strain_microstrain": strains.std(),
                "q25_strain_microstrain": q25,
                "q75_strain_microstrain": q75,
                "iqr_strain_microstrain": q75 - q25,
                "n_tensile_strain_points": n_tensile,
                "n_compressive_strain_points": n_compressive,
                "n_zero_strain_points": n_zero,
                "fraction_tensile_strain": n_tensile / n_retained_points,
                "fraction_compressive_strain": n_compressive / n_retained_points,
                "min_uncertainty_microstrain": uncertainties.min(),
                "max_uncertainty_microstrain": uncertainties.max(),
                "mean_uncertainty_microstrain": uncertainties.mean(),
                "median_uncertainty_microstrain": uncertainties.median(),
            }
        )

    summary = pd.DataFrame(rows)

    output_path = output_dir / "directional_strain_summary_table_expanded.csv"
    summary.to_csv(output_path, index=False)

    print("\nExpanded directional strain summary table:")
    print(summary.to_string(index=False))
    print(f"\nSaved expanded directional strain summary table to: {output_path}")

    return summary


def build_main_report_directional_strain_table(
    strain_summary: pd.DataFrame,
    output_dir: Path,
) -> pd.DataFrame:
    report = strain_summary.copy()

    report["retained_fraction_percent"] = report["retained_fraction"] * 100.0
    report["tensile_strain_points_percent"] = report["fraction_tensile_strain"] * 100.0

    report["min_location_mm"] = (
        "("
        + report["min_x_nom_mm"].round(3).astype(str)
        + ", "
        + report["min_y_nom_mm"].round(3).astype(str)
        + ")"
    )

    report["max_location_mm"] = (
        "("
        + report["max_x_nom_mm"].round(3).astype(str)
        + ", "
        + report["max_y_nom_mm"].round(3).astype(str)
        + ")"
    )

    main_cols = [
        "direction",
        "n_retained_points",
        "retained_fraction_percent",
        "min_strain_microstrain",
        "min_location_mm",
        "max_strain_microstrain",
        "max_location_mm",
        "uncertainty_at_max_microstrain",
        "mean_strain_microstrain",
        "median_strain_microstrain",
        "tensile_strain_points_percent",
    ]

    report = report[main_cols].copy()

    output_path = output_dir / "main_report_directional_strain_summary_table.csv"
    report.to_csv(output_path, index=False)

    print("\nMain-report directional strain summary table:")
    print(report.to_string(index=False))
    print(f"\nSaved main-report directional strain summary table to: {output_path}")

    return report


# ============================================================
# 10. Stress summary tables
# ============================================================

def build_stress_summary_table(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    retained_df = df[df["used_for_interpretation"]].copy()

    if retained_df.empty:
        raise ValueError("No retained interpretation points available for summary table.")

    n_total_analysis_points = len(df)
    n_retained_points = len(retained_df)
    n_rejected_points = n_total_analysis_points - n_retained_points
    retained_fraction = n_retained_points / n_total_analysis_points

    components = [
        {"component": "Longitudinal", "stress_col": "sig_long_py", "unc_col": "dsig_long_py"},
        {"component": "Transverse", "stress_col": "sig_trans_py", "unc_col": "dsig_trans_py"},
        {"component": "Normal", "stress_col": "sig_norm_py", "unc_col": "dsig_norm_py"},
    ]

    rows = []

    for item in components:
        component = item["component"]
        stress_col = item["stress_col"]
        unc_col = item["unc_col"]

        stresses = retained_df[stress_col]
        uncertainties = retained_df[unc_col]

        max_idx = stresses.idxmax()
        min_idx = stresses.idxmin()
        max_abs_idx = stresses.abs().idxmax()

        max_row = retained_df.loc[max_idx]
        min_row = retained_df.loc[min_idx]
        max_abs_row = retained_df.loc[max_abs_idx]

        n_tensile = (stresses > 0).sum()
        n_compressive = (stresses < 0).sum()
        n_zero = (stresses == 0).sum()

        q25 = stresses.quantile(0.25)
        q75 = stresses.quantile(0.75)

        rows.append(
            {
                "component": component,
                "n_total_analysis_points": n_total_analysis_points,
                "n_retained_points": n_retained_points,
                "n_rejected_points": n_rejected_points,
                "retained_fraction": retained_fraction,
                "min_stress_MPa": stresses.min(),
                "min_x_nom_mm": min_row["x_nom"],
                "min_y_nom_mm": min_row["y_nom"],
                "uncertainty_at_min_MPa": min_row[unc_col],
                "max_stress_MPa": stresses.max(),
                "max_x_nom_mm": max_row["x_nom"],
                "max_y_nom_mm": max_row["y_nom"],
                "uncertainty_at_max_MPa": max_row[unc_col],
                "max_abs_stress_MPa": max_abs_row[stress_col],
                "max_abs_stress_magnitude_MPa": abs(max_abs_row[stress_col]),
                "max_abs_x_nom_mm": max_abs_row["x_nom"],
                "max_abs_y_nom_mm": max_abs_row["y_nom"],
                "uncertainty_at_max_abs_MPa": max_abs_row[unc_col],
                "stress_range_MPa": stresses.max() - stresses.min(),
                "mean_stress_MPa": stresses.mean(),
                "median_stress_MPa": stresses.median(),
                "std_stress_MPa": stresses.std(),
                "q25_stress_MPa": q25,
                "q75_stress_MPa": q75,
                "iqr_stress_MPa": q75 - q25,
                "n_tensile_points": n_tensile,
                "n_compressive_points": n_compressive,
                "n_zero_points": n_zero,
                "fraction_tensile": n_tensile / n_retained_points,
                "fraction_compressive": n_compressive / n_retained_points,
                "min_uncertainty_MPa": uncertainties.min(),
                "max_uncertainty_MPa": uncertainties.max(),
                "mean_uncertainty_MPa": uncertainties.mean(),
                "median_uncertainty_MPa": uncertainties.median(),
            }
        )

    summary = pd.DataFrame(rows)

    output_path = output_dir / "retained_stress_summary_table_expanded.csv"
    summary.to_csv(output_path, index=False)

    print("\nExpanded retained stress summary table:")
    print(summary.to_string(index=False))
    print(f"\nSaved expanded retained stress summary table to: {output_path}")

    return summary


def build_main_report_summary_table(summary_df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    report = summary_df.copy()

    report["retained_fraction_percent"] = report["retained_fraction"] * 100.0

    report["min_location_mm"] = (
        "("
        + report["min_x_nom_mm"].round(3).astype(str)
        + ", "
        + report["min_y_nom_mm"].round(3).astype(str)
        + ")"
    )

    report["max_location_mm"] = (
        "("
        + report["max_x_nom_mm"].round(3).astype(str)
        + ", "
        + report["max_y_nom_mm"].round(3).astype(str)
        + ")"
    )

    report["tensile_points_percent"] = report["fraction_tensile"] * 100.0

    main_cols = [
        "component",
        "n_retained_points",
        "retained_fraction_percent",
        "min_stress_MPa",
        "min_location_mm",
        "max_stress_MPa",
        "max_location_mm",
        "uncertainty_at_max_MPa",
        "mean_stress_MPa",
        "median_stress_MPa",
        "tensile_points_percent",
    ]

    report = report[main_cols].copy()

    output_path = output_dir / "main_report_stress_summary_table.csv"
    report.to_csv(output_path, index=False)

    print("\nMain-report stress summary table:")
    print(report.to_string(index=False))
    print(f"\nSaved main-report stress summary table to: {output_path}")

    return report


def build_component_dominance_table(summary_df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    summary = summary_df.set_index("component")

    long_max = summary.loc["Longitudinal", "max_stress_MPa"]
    trans_max = summary.loc["Transverse", "max_stress_MPa"]
    norm_max = summary.loc["Normal", "max_stress_MPa"]

    long_mean = summary.loc["Longitudinal", "mean_stress_MPa"]
    trans_mean = summary.loc["Transverse", "mean_stress_MPa"]
    norm_mean = summary.loc["Normal", "mean_stress_MPa"]

    long_median = summary.loc["Longitudinal", "median_stress_MPa"]
    trans_median = summary.loc["Transverse", "median_stress_MPa"]
    norm_median = summary.loc["Normal", "median_stress_MPa"]

    rows = [
        {"comparison": "Max longitudinal / max transverse", "ratio": long_max / trans_max},
        {"comparison": "Max longitudinal / max normal", "ratio": long_max / norm_max},
        {"comparison": "Mean longitudinal / mean transverse", "ratio": long_mean / trans_mean},
        {"comparison": "Mean longitudinal / mean normal", "ratio": long_mean / norm_mean},
        {"comparison": "Median longitudinal / median transverse", "ratio": long_median / trans_median},
        {"comparison": "Median longitudinal / median normal", "ratio": long_median / norm_median},
    ]

    dominance = pd.DataFrame(rows)

    output_path = output_dir / "component_dominance_ratio_table.csv"
    dominance.to_csv(output_path, index=False)

    print("\nComponent dominance ratio table:")
    print(dominance.to_string(index=False))
    print(f"\nSaved component dominance ratio table to: {output_path}")

    return dominance


# ============================================================
# 11. Peak-location and curve-fit candidate screening
# ============================================================

def build_peak_by_y_table(df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    retained_df = df[df["used_for_interpretation"]].copy()

    if retained_df.empty:
        raise ValueError("No retained interpretation points available for peak-by-y table.")

    rows = []

    for y_value, group in retained_df.groupby("y_nom"):
        group = group.sort_values("x_nom").copy()

        n_points = len(group)

        max_idx = group["sig_long_py"].idxmax()
        min_idx = group["sig_long_py"].idxmin()

        max_row = group.loc[max_idx]
        min_row = group.loc[min_idx]

        x_min = group["x_nom"].min()
        x_max = group["x_nom"].max()

        has_negative_x = (group["x_nom"] < 0).any()
        has_positive_x = (group["x_nom"] > 0).any()
        has_zero_x = (group["x_nom"] == 0).any()

        rows.append(
            {
                "y_nom_mm": y_value,
                "n_retained_points": n_points,
                "x_min_mm": x_min,
                "x_max_mm": x_max,
                "has_negative_x": has_negative_x,
                "has_zero_x": has_zero_x,
                "has_positive_x": has_positive_x,
                "max_longitudinal_stress_MPa": max_row["sig_long_py"],
                "x_at_max_longitudinal_mm": max_row["x_nom"],
                "uncertainty_at_max_longitudinal_MPa": max_row["dsig_long_py"],
                "min_longitudinal_stress_MPa": min_row["sig_long_py"],
                "x_at_min_longitudinal_mm": min_row["x_nom"],
                "uncertainty_at_min_longitudinal_MPa": min_row["dsig_long_py"],
                "longitudinal_stress_range_MPa": (
                    group["sig_long_py"].max() - group["sig_long_py"].min()
                ),
                "mean_longitudinal_stress_MPa": group["sig_long_py"].mean(),
                "median_longitudinal_stress_MPa": group["sig_long_py"].median(),
            }
        )

    peak_table = pd.DataFrame(rows).sort_values("y_nom_mm").reset_index(drop=True)

    output_path = output_dir / "measured_longitudinal_peak_by_y_table.csv"
    peak_table.to_csv(output_path, index=False)

    print("\nMeasured longitudinal peak-by-y table:")
    print(peak_table.to_string(index=False))
    print(f"\nSaved measured longitudinal peak-by-y table to: {output_path}")

    return peak_table


def build_fit_candidate_table(
    peak_table: pd.DataFrame,
    output_dir: Path,
    min_points_for_fit: int = 6,
) -> pd.DataFrame:
    candidates = peak_table.copy()

    candidates["sufficient_points_for_fit"] = (
        candidates["n_retained_points"] >= min_points_for_fit
    )

    candidates["has_two_sided_x_coverage"] = (
        candidates["has_negative_x"]
        & candidates["has_zero_x"]
        & candidates["has_positive_x"]
    )

    candidates["fit_candidate"] = (
        candidates["sufficient_points_for_fit"]
        & candidates["has_two_sided_x_coverage"]
    )

    output_path = output_dir / "curve_fit_candidate_y_lines.csv"
    candidates.to_csv(output_path, index=False)

    print("\nCurve-fit candidate y-lines:")
    print(candidates.to_string(index=False))
    print(f"\nSaved curve-fit candidate table to: {output_path}")

    return candidates


# ============================================================
# 11B. Local quadratic curve fitting for y = -15 mm
# ============================================================

def _select_local_window_around_measured_peak(
    group: pd.DataFrame,
    stress_col: str = "sig_long_py",
    points_each_side: int = 3,
    min_points: int = 5,
) -> pd.DataFrame:
    group = group.sort_values("x_nom").copy().reset_index(drop=True)

    if group.empty:
        return group

    peak_pos = int(group[stress_col].idxmax())

    start = max(0, peak_pos - points_each_side)
    end = min(len(group), peak_pos + points_each_side + 1)

    while (end - start) < min_points and (start > 0 or end < len(group)):
        if start > 0:
            start -= 1
        if (end - start) >= min_points:
            break
        if end < len(group):
            end += 1

    return group.iloc[start:end].copy()


def _fit_quadratic_peak(
    x: np.ndarray,
    y: np.ndarray,
    yerr: np.ndarray | None,
    model_label: str,
) -> dict:
    result = {
        "model": model_label,
        "fit_status": "not_run",
        "n_fit_points": len(x),
        "quadratic_a": np.nan,
        "quadratic_b": np.nan,
        "quadratic_c": np.nan,
        "fitted_peak_x_mm": np.nan,
        "fitted_peak_stress_MPa": np.nan,
        "vertex_inside_fit_range": False,
        "opens_downward": False,
        "fit_rmse_MPa": np.nan,
        "weighted_rmse": np.nan,
        "fit_x_min_mm": np.nan,
        "fit_x_max_mm": np.nan,
    }

    if len(x) < 3 or len(np.unique(x)) < 3:
        result["fit_status"] = "failed_insufficient_unique_points"
        return result

    result["fit_x_min_mm"] = float(np.min(x))
    result["fit_x_max_mm"] = float(np.max(x))

    try:
        if yerr is not None:
            valid_unc = np.isfinite(yerr) & (yerr > 0)

            if valid_unc.sum() != len(yerr):
                result["fit_status"] = "failed_invalid_uncertainty_values"
                return result

            weights = 1.0 / yerr
            coeffs = np.polyfit(x, y, deg=2, w=weights)
        else:
            coeffs = np.polyfit(x, y, deg=2)

        a, b, c = coeffs

        result["quadratic_a"] = float(a)
        result["quadratic_b"] = float(b)
        result["quadratic_c"] = float(c)

        y_hat = np.polyval(coeffs, x)
        residuals = y - y_hat
        result["fit_rmse_MPa"] = float(np.sqrt(np.mean(residuals**2)))

        if yerr is not None:
            result["weighted_rmse"] = float(np.sqrt(np.mean((residuals / yerr) ** 2)))

        if abs(a) < 1.0e-12:
            result["fit_status"] = "failed_nearly_linear_fit"
            return result

        x_vertex = -b / (2.0 * a)
        y_vertex = np.polyval(coeffs, x_vertex)

        result["fitted_peak_x_mm"] = float(x_vertex)
        result["fitted_peak_stress_MPa"] = float(y_vertex)
        result["vertex_inside_fit_range"] = bool(np.min(x) <= x_vertex <= np.max(x))
        result["opens_downward"] = bool(a < 0)

        if a < 0 and result["vertex_inside_fit_range"]:
            result["fit_status"] = "valid_local_maximum"
        elif a >= 0:
            result["fit_status"] = "invalid_opens_upward_no_local_maximum"
        else:
            result["fit_status"] = "invalid_vertex_outside_fit_range"

        return result

    except Exception as exc:
        result["fit_status"] = f"failed_exception_{type(exc).__name__}"
        return result


def add_curve_fit_quality_flags(
    fit_results: pd.DataFrame,
    fit_points: pd.DataFrame,
    measured_peak_stress_mpa: float,
    measured_peak_uncertainty_mpa: float,
    location_tolerance_factor: float = 1.0,
    magnitude_relative_tolerance: float = 0.10,
    uncertainty_multiplier: float = 2.0,
    rmse_relative_tolerance: float = 0.20,
) -> pd.DataFrame:
    """
    Add interpretation flags to local quadratic curve-fit results.

    These project-specific diagnostic criteria assess whether the fitted curve
    supports peak location and/or peak magnitude. The measured peak remains
    the primary result.
    """
    out = fit_results.copy()

    unique_x = np.sort(fit_points["x_nom"].dropna().unique())

    if len(unique_x) >= 2:
        median_spacing = float(np.median(np.diff(unique_x)))
    else:
        median_spacing = np.nan

    location_tolerance_mm = location_tolerance_factor * median_spacing

    magnitude_tolerance_mpa = max(
        uncertainty_multiplier * measured_peak_uncertainty_mpa,
        magnitude_relative_tolerance * abs(measured_peak_stress_mpa),
    )

    rmse_tolerance_mpa = rmse_relative_tolerance * abs(measured_peak_stress_mpa)

    out["median_fit_point_spacing_mm"] = median_spacing
    out["peak_location_tolerance_mm"] = location_tolerance_mm
    out["peak_magnitude_tolerance_MPa"] = magnitude_tolerance_mpa
    out["rmse_tolerance_MPa"] = rmse_tolerance_mpa

    out["abs_delta_fitted_minus_measured_x_mm"] = (
        out["delta_fitted_minus_measured_x_mm"].abs()
    )

    out["abs_delta_fitted_minus_measured_stress_MPa"] = (
        out["delta_fitted_minus_measured_stress_MPa"].abs()
    )

    out["acceptable_for_peak_location"] = (
        (out["fit_status"] == "valid_local_maximum")
        & out["vertex_inside_fit_range"]
        & out["opens_downward"]
        & (out["abs_delta_fitted_minus_measured_x_mm"] <= location_tolerance_mm)
    )

    out["acceptable_for_peak_magnitude"] = (
        out["acceptable_for_peak_location"]
        & (out["abs_delta_fitted_minus_measured_stress_MPa"] <= magnitude_tolerance_mpa)
        & (out["fit_rmse_MPa"] <= rmse_tolerance_mpa)
    )

    def classify_fit(row: pd.Series) -> str:
        if row["acceptable_for_peak_magnitude"]:
            return "supports_peak_location_and_magnitude"
        if row["acceptable_for_peak_location"]:
            return "supports_peak_location_only"
        if row["fit_status"] != "valid_local_maximum":
            return "not_reliable_as_peak_fit"
        return "diagnostic_only"

    out["fit_interpretation_class"] = out.apply(classify_fit, axis=1)

    out["method_recommendation"] = np.where(
        out["acceptable_for_peak_magnitude"],
        "Fitted peak may be reported as supplementary estimate; measured peak remains primary.",
        "Use measured peak as primary result; fitted curve is diagnostic only.",
    )

    return out


def build_y_minus15_local_quadratic_fit(
    df: pd.DataFrame,
    output_dir: Path,
    target_y: float = -15.0,
    points_each_side: int = 3,
    min_fit_points: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    line_df = df[
        df["used_for_interpretation"]
        & np.isclose(df["y_nom"], target_y)
    ].copy()

    line_df = line_df.dropna(
        subset=["x_nom", "sig_long_py", "dsig_long_py"]
    ).sort_values("x_nom")

    if line_df.empty:
        raise ValueError(f"No retained interpretation points found for y = {target_y} mm.")

    measured_peak_idx = line_df["sig_long_py"].idxmax()
    measured_peak = line_df.loc[measured_peak_idx]

    fit_points = _select_local_window_around_measured_peak(
        line_df,
        stress_col="sig_long_py",
        points_each_side=points_each_side,
        min_points=min_fit_points,
    )

    fit_points_output = fit_points.copy()
    fit_points_output["target_y_mm"] = target_y
    fit_points_output["used_in_local_quadratic_fit"] = True

    x = fit_points["x_nom"].to_numpy(dtype=float)
    y = fit_points["sig_long_py"].to_numpy(dtype=float)
    yerr = fit_points["dsig_long_py"].to_numpy(dtype=float)

    unweighted = _fit_quadratic_peak(
        x=x,
        y=y,
        yerr=None,
        model_label="local_quadratic_unweighted",
    )

    weighted = _fit_quadratic_peak(
        x=x,
        y=y,
        yerr=yerr,
        model_label="local_quadratic_weighted_by_dsig_long",
    )

    rows = []

    for fit_result in [unweighted, weighted]:
        row = {
            **fit_result,
            "target_y_mm": target_y,
            "measured_peak_x_mm": float(measured_peak["x_nom"]),
            "measured_peak_stress_MPa": float(measured_peak["sig_long_py"]),
            "measured_peak_uncertainty_MPa": float(measured_peak["dsig_long_py"]),
            "measured_peak_excel_row": measured_peak.get("excel_row", np.nan),
            "delta_fitted_minus_measured_x_mm": (
                fit_result["fitted_peak_x_mm"] - float(measured_peak["x_nom"])
                if pd.notna(fit_result["fitted_peak_x_mm"])
                else np.nan
            ),
            "delta_fitted_minus_measured_stress_MPa": (
                fit_result["fitted_peak_stress_MPa"] - float(measured_peak["sig_long_py"])
                if pd.notna(fit_result["fitted_peak_stress_MPa"])
                else np.nan
            ),
            "interpretation_note": (
                "Measured peak remains primary; fitted peak is a local quadratic estimate."
            ),
        }

        rows.append(row)

    fit_results = pd.DataFrame(rows)

    fit_results = add_curve_fit_quality_flags(
        fit_results=fit_results,
        fit_points=fit_points,
        measured_peak_stress_mpa=float(measured_peak["sig_long_py"]),
        measured_peak_uncertainty_mpa=float(measured_peak["dsig_long_py"]),
        location_tolerance_factor=1.0,
        magnitude_relative_tolerance=0.10,
        uncertainty_multiplier=2.0,
        rmse_relative_tolerance=0.20,
    )

    results_path = output_dir / "curve_fit_y_minus15_local_quadratic_results.csv"
    points_path = output_dir / "curve_fit_y_minus15_fit_points.csv"

    fit_results.to_csv(results_path, index=False)
    fit_points_output.to_csv(points_path, index=False)

    print("\nLocal quadratic fit results for y = -15 mm:")
    print(fit_results.to_string(index=False))
    print(f"\nSaved curve-fit results to: {results_path}")
    print(f"Saved curve-fit input points to: {points_path}")

    plot_y_minus15_local_quadratic_fit(
        line_df=line_df,
        fit_points=fit_points,
        fit_results=fit_results,
        output_dir=output_dir,
        target_y=target_y,
    )

    return fit_results, fit_points_output


def plot_y_minus15_local_quadratic_fit(
    line_df: pd.DataFrame,
    fit_points: pd.DataFrame,
    fit_results: pd.DataFrame,
    output_dir: Path,
    target_y: float = -15.0,
) -> None:
    plt.figure(figsize=(9, 5.5))

    plt.errorbar(
        line_df["x_nom"],
        line_df["sig_long_py"],
        yerr=line_df["dsig_long_py"],
        marker="o",
        linestyle="none",
        capsize=3,
        label="Retained measured points",
    )

    plt.scatter(
        fit_points["x_nom"],
        fit_points["sig_long_py"],
        s=110,
        marker="s",
        facecolors="none",
        label="Local fit points",
    )

    measured_peak = line_df.loc[line_df["sig_long_py"].idxmax()]

    plt.axvline(
        measured_peak["x_nom"],
        linestyle="--",
        label="Measured peak location",
    )

    x_min = fit_points["x_nom"].min()
    x_max = fit_points["x_nom"].max()
    x_dense = np.linspace(x_min, x_max, 300)

    for _, row in fit_results.iterrows():
        if row["fit_status"] == "valid_local_maximum":
            coeffs = [
                row["quadratic_a"],
                row["quadratic_b"],
                row["quadratic_c"],
            ]

            y_dense = np.polyval(coeffs, x_dense)

            plt.plot(
                x_dense,
                y_dense,
                label=row["model"],
            )

            plt.scatter(
                row["fitted_peak_x_mm"],
                row["fitted_peak_stress_MPa"],
                marker="^",
                s=100,
                label=f"Fitted peak: {row['model']}",
            )

    plt.xlabel("Nominal x-position, x (mm)")
    plt.ylabel("Longitudinal stress, σL (MPa)")
    plt.title(f"Local quadratic peak estimation along y = {target_y:g} mm")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    output_path = output_dir / "curve_fit_y_minus15_local_quadratic.png"
    plt.savefig(output_path, dpi=300)
    plt.close()

    print(f"Saved curve-fit figure to: {output_path}")


# ============================================================
# 12. 1D diagnostic plots
# ============================================================

def plot_stress_vs_x(df: pd.DataFrame, output_dir: Path) -> None:
    plot_df = df[df["used_for_interpretation"]].copy()

    if plot_df.empty:
        print("Skipping stress-vs-x plots: no retained interpretation points.")
        return

    plot_specs = [
        ("sig_long_py", "Longitudinal Stress vs x", "longitudinal_stress_vs_x.png"),
        ("sig_trans_py", "Transverse Stress vs x", "transverse_stress_vs_x.png"),
        ("sig_norm_py", "Normal Stress vs x", "normal_stress_vs_x.png"),
    ]

    for stress_col, title, filename in plot_specs:
        plt.figure(figsize=(8, 5))
        plt.scatter(plot_df["x_nom"], plot_df[stress_col], s=70)
        plt.xlabel("Nominal x-position, x (mm)")
        plt.ylabel("Stress (MPa)")
        plt.title(title)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(output_dir / filename, dpi=300)
        plt.close()


def plot_selected_y_line(
    df: pd.DataFrame,
    y_value: float,
    output_dir: Path,
    min_points: int = 3,
) -> None:
    subset = df[
        (df["y_nom"] == y_value)
        & (df["used_for_interpretation"])
    ].sort_values("x_nom").copy()

    if len(subset) < min_points:
        print(
            f"Skipping y_nom = {y_value}: only {len(subset)} interpretable point(s) remain."
        )
        return

    plt.figure(figsize=(9, 5.5))

    plt.errorbar(
        subset["x_nom"], subset["sig_long_py"],
        yerr=subset["dsig_long_py"],
        marker="o", capsize=3, label="Longitudinal",
    )

    plt.errorbar(
        subset["x_nom"], subset["sig_trans_py"],
        yerr=subset["dsig_trans_py"],
        marker="o", capsize=3, label="Transverse",
    )

    plt.errorbar(
        subset["x_nom"], subset["sig_norm_py"],
        yerr=subset["dsig_norm_py"],
        marker="o", capsize=3, label="Normal",
    )

    plt.xlabel("Nominal x-position, x (mm)")
    plt.ylabel("Stress (MPa)")
    plt.title(f"Stress components along y = {y_value} mm after filtering")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / f"stress_components_y_{y_value}_filtered_errorbars.png", dpi=300)
    plt.close()


# ============================================================
# 13. 2D point-based maps
# ============================================================

def plot_stress_location_map(
    df: pd.DataFrame,
    stress_col: str,
    title: str,
    filename: str,
    output_dir: Path,
    use_interpretation_filter: bool = True,
) -> None:
    if use_interpretation_filter:
        plot_df = df[df["used_for_interpretation"]].copy()
    else:
        plot_df = df[df["used_for_plotting"]].copy()

    if plot_df.empty:
        print(f"Skipping {filename}: no points available.")
        return

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        plot_df["x_nom"],
        plot_df["y_nom"],
        c=plot_df[stress_col],
        s=90,
    )

    plt.xlabel("Nominal x-position, x (mm)")
    plt.ylabel("Nominal y-position, y (mm)")
    plt.title(title)
    plt.grid(True)
    plt.colorbar(sc, label="Stress (MPa)")
    plt.tight_layout()
    plt.savefig(output_dir / filename, dpi=300)
    plt.close()


def plot_strain_location_map(
    df: pd.DataFrame,
    strain_col: str,
    title: str,
    filename: str,
    output_dir: Path,
) -> None:
    plot_df = df[df["used_for_interpretation"]].copy()

    if plot_df.empty:
        print(f"Skipping {filename}: no retained points available.")
        return

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        plot_df["x_nom"],
        plot_df["y_nom"],
        c=plot_df[strain_col],
        s=90,
    )

    plt.xlabel("Nominal x-position, x (mm)")
    plt.ylabel("Nominal y-position, y (mm)")
    plt.title(title)
    plt.grid(True)
    plt.colorbar(sc, label="Strain (microstrain)")
    plt.tight_layout()
    plt.savefig(output_dir / filename, dpi=300)
    plt.close()


def plot_uncertainty_location_map(
    df: pd.DataFrame,
    unc_col: str,
    title: str,
    filename: str,
    output_dir: Path,
) -> None:
    plot_df = df[df["used_for_plotting"]].copy()

    if plot_df.empty:
        print(f"Skipping {filename}: no plottable points available.")
        return

    plt.figure(figsize=(8, 6))
    sc = plt.scatter(
        plot_df["x_nom"],
        plot_df["y_nom"],
        c=plot_df[unc_col],
        s=90,
    )

    plt.xlabel("Nominal x-position, x (mm)")
    plt.ylabel("Nominal y-position, y (mm)")
    plt.title(title)
    plt.grid(True)
    plt.colorbar(sc, label="Stress uncertainty (MPa)")
    plt.tight_layout()
    plt.savefig(output_dir / filename, dpi=300)
    plt.close()


def plot_retained_vs_rejected_points(df: pd.DataFrame, output_dir: Path) -> None:
    retained = df[df["used_for_interpretation"]].copy()
    rejected = df[~df["used_for_interpretation"]].copy()

    plt.figure(figsize=(8, 6))

    if not retained.empty:
        plt.scatter(
            retained["x_nom"], retained["y_nom"],
            s=90, marker="o", label="Retained for interpretation",
        )

    if not rejected.empty:
        plt.scatter(
            rejected["x_nom"], rejected["y_nom"],
            s=110, marker="x", label="Rejected from interpretation",
        )

    plt.xlabel("Nominal x-position, x (mm)")
    plt.ylabel("Nominal y-position, y (mm)")
    plt.title("Retained and rejected measurement points")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / "retained_vs_rejected_points.png", dpi=300)
    plt.close()


# ============================================================
# 14. Interpolation helpers
# ============================================================

def get_interpolation_grid(
    plot_df: pd.DataFrame,
    stress_col: str,
    grid_resolution: int = 250,
    method: str = "linear",
):
    x = plot_df["x_nom"].to_numpy()
    y = plot_df["y_nom"].to_numpy()
    z = plot_df[stress_col].to_numpy()

    xi = np.linspace(x.min(), x.max(), grid_resolution)
    yi = np.linspace(y.min(), y.max(), grid_resolution)
    xi_grid, yi_grid = np.meshgrid(xi, yi)

    zi_grid = griddata(
        points=(x, y),
        values=z,
        xi=(xi_grid, yi_grid),
        method=method,
    )

    return x, y, xi_grid, yi_grid, zi_grid


def plot_interpolated_stress_map(
    df: pd.DataFrame,
    stress_col: str,
    title: str,
    filename: str,
    output_dir: Path,
    use_interpretation_filter: bool = True,
    grid_resolution: int = 250,
    method: str = "linear",
) -> None:
    if use_interpretation_filter:
        plot_df = df[df["used_for_interpretation"]].copy()
        point_label = "Retained points"
    else:
        plot_df = df[df["used_for_plotting"]].copy()
        point_label = "All structurally valid points"

    plot_df = plot_df.dropna(subset=["x_nom", "y_nom", stress_col])

    if len(plot_df) < 3:
        print(f"Skipping {filename}: fewer than 3 points available for interpolation.")
        return

    x, y, xi_grid, yi_grid, zi_grid = get_interpolation_grid(
        plot_df,
        stress_col,
        grid_resolution=grid_resolution,
        method=method,
    )

    plt.figure(figsize=(8, 6))

    contour = plt.contourf(
        xi_grid,
        yi_grid,
        zi_grid,
        levels=20,
    )

    plt.scatter(
        x,
        y,
        s=35,
        edgecolors="black",
        facecolors="none",
        label=point_label,
    )

    plt.xlabel("Nominal x-position, x (mm)")
    plt.ylabel("Nominal y-position, y (mm)")
    plt.title(title)
    plt.colorbar(contour, label="Stress (MPa)")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_dir / filename, dpi=300)
    plt.close()


def plot_interpolation_comparison(
    df: pd.DataFrame,
    stress_col: str,
    component_name: str,
    output_dir: Path,
) -> None:
    plot_interpolated_stress_map(
        df,
        stress_col=stress_col,
        title=f"Unfiltered interpolated {component_name} stress map",
        filename=f"interpolated_{component_name}_stress_unfiltered.png",
        output_dir=output_dir,
        use_interpretation_filter=False,
    )

    plot_interpolated_stress_map(
        df,
        stress_col=stress_col,
        title=f"Controlled interpolated {component_name} stress map",
        filename=f"interpolated_{component_name}_stress_filtered.png",
        output_dir=output_dir,
        use_interpretation_filter=True,
    )


def plot_side_by_side_interpolation_comparison(
    df: pd.DataFrame,
    stress_col: str,
    component_name: str,
    output_dir: Path,
    grid_resolution: int = 250,
    method: str = "linear",
) -> None:
    unfiltered = df[df["used_for_plotting"]].copy().dropna(subset=["x_nom", "y_nom", stress_col])
    filtered = df[df["used_for_interpretation"]].copy().dropna(subset=["x_nom", "y_nom", stress_col])

    if len(unfiltered) < 3 or len(filtered) < 3:
        print(f"Skipping side-by-side {component_name}: insufficient points.")
        return

    vmin = min(unfiltered[stress_col].min(), filtered[stress_col].min())
    vmax = max(unfiltered[stress_col].max(), filtered[stress_col].max())

    x_u, y_u, xi_u, yi_u, zi_u = get_interpolation_grid(
        unfiltered,
        stress_col,
        grid_resolution=grid_resolution,
        method=method,
    )

    x_f, y_f, xi_f, yi_f, zi_f = get_interpolation_grid(
        filtered,
        stress_col,
        grid_resolution=grid_resolution,
        method=method,
    )

    levels = np.linspace(vmin, vmax, 21)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

    contour_u = axes[0].contourf(
        xi_u,
        yi_u,
        zi_u,
        levels=levels,
        vmin=vmin,
        vmax=vmax,
    )

    axes[0].scatter(
        x_u,
        y_u,
        s=30,
        edgecolors="black",
        facecolors="none",
        label="All structurally valid points",
    )

    axes[0].set_title(f"Unfiltered {component_name} stress")
    axes[0].set_xlabel("Nominal x-position, x (mm)")
    axes[0].set_ylabel("Nominal y-position, y (mm)")
    axes[0].grid(True)
    axes[0].legend()

    contour_f = axes[1].contourf(
        xi_f,
        yi_f,
        zi_f,
        levels=levels,
        vmin=vmin,
        vmax=vmax,
    )

    axes[1].scatter(
        x_f,
        y_f,
        s=30,
        edgecolors="black",
        facecolors="none",
        label="Retained points",
    )

    axes[1].set_title(f"Controlled {component_name} stress")
    axes[1].set_xlabel("Nominal x-position, x (mm)")
    axes[1].set_ylabel("Nominal y-position, y (mm)")
    axes[1].grid(True)
    axes[1].legend()

    cbar = fig.colorbar(contour_f, ax=axes, shrink=0.9)
    cbar.set_label("Stress (MPa)")

    fig.suptitle(
        f"Effect of uncertainty filtering on interpolated {component_name} stress field",
        fontsize=15,
    )

    fig.savefig(
        output_dir / f"comparison_interpolated_{component_name}_stress_common_scale.png",
        dpi=300,
    )

    plt.close(fig)


# ============================================================
# 15. Main
# ============================================================

def main() -> None:
    df_input = load_full_samed0(EXCEL_FILE)

    print_structural_summary(df_input)
    inspect_point_structure(df_input)
    inspect_duplicate_content(df_input)

    df_results = run_pipeline(df_input)
    df_validated = add_validation_columns(df_results)

    print_validation_summary(df_validated)

    validation_output = OUTPUT_DIR / "step1_full_samed0_python_validation.csv"
    df_validated.to_csv(validation_output, index=False)
    print(f"\nSaved validation table to: {validation_output}")

    df_analysis = build_analysis_dataset(df_validated)
    df_analysis = add_quality_flags(df_analysis)

    print_quality_flag_summary(df_analysis)
    inspect_catastrophic_rows(df_analysis)
    print_interpretable_points_by_y(df_analysis)

    export_missing_data_reports(df_analysis, OUTPUT_DIR)

    reference_table = build_reference_value_comparison_table(df_analysis, OUTPUT_DIR)
    build_main_report_reference_table(reference_table, OUTPUT_DIR)

    strain_summary = build_directional_strain_summary_table(df_analysis, OUTPUT_DIR)
    build_main_report_directional_strain_table(strain_summary, OUTPUT_DIR)

    expanded_summary = build_stress_summary_table(df_analysis, OUTPUT_DIR)
    build_main_report_summary_table(expanded_summary, OUTPUT_DIR)
    build_component_dominance_table(expanded_summary, OUTPUT_DIR)

    peak_by_y_table = build_peak_by_y_table(df_analysis, OUTPUT_DIR)
    build_fit_candidate_table(peak_by_y_table, OUTPUT_DIR, min_points_for_fit=6)

    build_y_minus15_local_quadratic_fit(
        df_analysis,
        OUTPUT_DIR,
        target_y=-15.0,
        points_each_side=3,
        min_fit_points=5,
    )

    export_rejected_points(df_analysis, OUTPUT_DIR)

    analysis_output = OUTPUT_DIR / "step1_full_samed0_analysis_dataset.csv"
    df_analysis.to_csv(analysis_output, index=False)
    print(f"\nSaved analysis dataset to: {analysis_output}")

    print("\nAnalysis dataset summary:")
    print(f"  Rows in validation dataset: {len(df_validated)}")
    print(f"  Rows in analysis dataset:   {len(df_analysis)}")

    plot_stress_vs_x(df_analysis, OUTPUT_DIR)

    plot_selected_y_line(df_analysis, y_value=-15, output_dir=OUTPUT_DIR)
    plot_selected_y_line(df_analysis, y_value=-27, output_dir=OUTPUT_DIR)
    plot_selected_y_line(df_analysis, y_value=-3, output_dir=OUTPUT_DIR)

    plot_strain_location_map(
        df_analysis,
        strain_col="eps_long_py",
        title="Longitudinal strain-location map (filtered)",
        filename="longitudinal_strain_location_map_filtered.png",
        output_dir=OUTPUT_DIR,
    )

    plot_strain_location_map(
        df_analysis,
        strain_col="eps_trans_py",
        title="Transverse strain-location map (filtered)",
        filename="transverse_strain_location_map_filtered.png",
        output_dir=OUTPUT_DIR,
    )

    plot_strain_location_map(
        df_analysis,
        strain_col="eps_norm_py",
        title="Normal strain-location map (filtered)",
        filename="normal_strain_location_map_filtered.png",
        output_dir=OUTPUT_DIR,
    )

    plot_stress_location_map(
        df_analysis,
        stress_col="sig_long_py",
        title="Longitudinal stress-location map (filtered)",
        filename="longitudinal_stress_location_map_filtered.png",
        output_dir=OUTPUT_DIR,
        use_interpretation_filter=True,
    )

    plot_stress_location_map(
        df_analysis,
        stress_col="sig_trans_py",
        title="Transverse stress-location map (filtered)",
        filename="transverse_stress_location_map_filtered.png",
        output_dir=OUTPUT_DIR,
        use_interpretation_filter=True,
    )

    plot_stress_location_map(
        df_analysis,
        stress_col="sig_norm_py",
        title="Normal stress-location map (filtered)",
        filename="normal_stress_location_map_filtered.png",
        output_dir=OUTPUT_DIR,
        use_interpretation_filter=True,
    )

    plot_uncertainty_location_map(
        df_analysis,
        unc_col="dsig_long_py",
        title="Longitudinal stress uncertainty-location map (all plottable points)",
        filename="longitudinal_uncertainty_location_map_all_points.png",
        output_dir=OUTPUT_DIR,
    )

    plot_retained_vs_rejected_points(df_analysis, OUTPUT_DIR)

    plot_interpolation_comparison(
        df_analysis,
        stress_col="sig_long_py",
        component_name="longitudinal",
        output_dir=OUTPUT_DIR,
    )

    plot_interpolation_comparison(
        df_analysis,
        stress_col="sig_trans_py",
        component_name="transverse",
        output_dir=OUTPUT_DIR,
    )

    plot_interpolation_comparison(
        df_analysis,
        stress_col="sig_norm_py",
        component_name="normal",
        output_dir=OUTPUT_DIR,
    )

    plot_side_by_side_interpolation_comparison(
        df_analysis,
        stress_col="sig_long_py",
        component_name="longitudinal",
        output_dir=OUTPUT_DIR,
    )

    plot_side_by_side_interpolation_comparison(
        df_analysis,
        stress_col="sig_trans_py",
        component_name="transverse",
        output_dir=OUTPUT_DIR,
    )

    plot_side_by_side_interpolation_comparison(
        df_analysis,
        stress_col="sig_norm_py",
        component_name="normal",
        output_dir=OUTPUT_DIR,
    )

    print(f"\nSaved all results to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()